# Module 1 · What is an API, and How Do You Authenticate to One?
### FastAPI for ML Engineers — Bootcamp Session (Part 1 of 4)

**Time budget:** ~60 minutes
**Format:** live-coded, fully runnable — no external server needed (we use FastAPI's `TestClient`)

## Agenda
| Time | Topic |
|---|---|
| 0:00–0:10 | What is an API? Client/server, HTTP, REST |
| 0:10–0:20 | Anatomy of an HTTP request/response |
| 0:20–0:35 | Your first FastAPI app: routes, path & query params |
| 0:35–0:55 | Authentication: API keys, HTTP Basic, OAuth2 + JWT |
| 0:55–1:00 | Exercise + wrap-up |

## Learning objectives
By the end of this notebook you will be able to:
- Explain what an API is in terms an ML engineer actually needs (not just "it's an interface")
- Read and write HTTP requests/responses with confidence
- Stand up a minimal FastAPI service
- Protect an endpoint with an API key, HTTP Basic auth, and OAuth2/JWT — the three schemes you'll meet 90% of the time when serving models

## 1.1 What is an API?

An **API (Application Programming Interface)** is a contract that lets one piece of software ask another piece of software to do something, without needing to know how it's implemented internally.

For an ML engineer, the API is almost always the **boundary between your model and the rest of the world**:

```
[ Client / App / Another Service ]  --request-->  [ Your API ]  --calls-->  [ Your Model ]
                                     <--response--
```

Think of it like a restaurant:
- **You (the client)** don't walk into the kitchen and cook your own food.
- **The waiter (the API)** takes your order in a fixed format (the menu = the API contract), passes it to the kitchen, and brings back a fixed-format response.
- **The kitchen (your model)** doesn't care who ordered — it just gets a well-formed ticket.

A **web API** specifically is an API exposed over HTTP, usually returning JSON. That's what we're building all session: a web API that wraps a model.

### Why this matters for ML specifically
Training a great model is maybe 30% of shipping ML. The rest is:
- Getting features to the model reliably
- Returning predictions in a format downstream systems can consume
- Versioning, auth, rate limiting, logging, monitoring
- Packaging it so it can scale (this is where Docker + Vertex AI come in, later modules)

FastAPI is the most common tool ML engineers reach for because it's fast to write, has automatic validation (via Pydantic), and generates interactive docs for free.

## 1.2 Anatomy of an HTTP request/response

Every HTTP interaction has the same shape, whether you're calling a weather API or your own fraud model.

**Request:**
```
POST /predict HTTP/1.1
Host: api.example.com
Content-Type: application/json
Authorization: Bearer <token>

{"feature_1": 0.42, "feature_2": 17}
```

**Response:**
```
HTTP/1.1 200 OK
Content-Type: application/json

{"prediction": 1, "probability": 0.87}
```

### Key parts
- **Method** — what you want to do (`GET`, `POST`, `PUT`, `DELETE`, ...)
- **Path** — which resource/endpoint (`/predict`)
- **Headers** — metadata (auth tokens, content type, request id)
- **Body** — the payload (usually JSON for ML APIs)
- **Status code** — did it work? (see table below)

### Common status codes you'll actually use
| Code | Meaning | When an ML API returns it |
|---|---|---|
| 200 | OK | Prediction returned successfully |
| 201 | Created | A new resource was created (e.g. a training job) |
| 400 | Bad Request | Input failed validation (wrong shape/type) |
| 401 | Unauthorized | Missing/invalid credentials |
| 403 | Forbidden | Valid credentials, but not allowed to do this |
| 404 | Not Found | Wrong path, or model/resource doesn't exist |
| 422 | Unprocessable Entity | FastAPI's default for Pydantic validation errors |
| 429 | Too Many Requests | Rate limit hit |
| 500 | Internal Server Error | Your code (or the model) threw an exception |
| 503 | Service Unavailable | Model not loaded yet / downstream dependency down |

In [ ]:
# Setup — run this once.
# (--break-system-packages is only needed on Debian/Ubuntu-managed Python installs;
#  harmless to leave in on Colab / venvs / conda too.)
!pip install -q --break-system-packages fastapi "uvicorn[standard]" python-multipart "python-jose[cryptography]" "passlib[bcrypt]" httpx requests scikit-learn joblib
print("done")

## 1.3 Your first FastAPI app

We'll build it up piece by piece. Notice three things FastAPI gives you almost for free:
1. **Automatic request validation** via Python type hints + Pydantic
2. **Automatic docs** at `/docs` (Swagger UI) and `/redoc`
3. **Async support** out of the box

> **A note on running this notebook:** instead of starting a real background server and hitting `localhost` (which is finicky inside notebooks), we use FastAPI's `TestClient`. It calls your app directly in-process — same routing, same validation, same responses — so every code cell below is fully runnable as-is. When you're not in a notebook, you'd run this with `uvicorn app:app --reload` from the terminal.

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="ML Serving API", version="0.1.0")

@app.get("/")
def root():
    return {"message": "ML serving API is up"}

@app.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

resp = client.get("/")
print(resp.status_code)
print(resp.json())

### Path parameters and query parameters

- **Path parameters** are part of the URL structure and identify *which* resource: `/models/{model_name}`
- **Query parameters** are optional modifiers appended after `?`: `/models/fraud_v2?version=3`

FastAPI infers which is which from your function signature — parameters that appear in the decorated path string are path params; everything else becomes a query param automatically.

In [ ]:
@app.get("/models/{model_name}")
def get_model_info(model_name: str, version: int = 1):
    # model_name -> path param (required, part of the URL)
    # version    -> query param (optional, defaults to 1)
    return {"model_name": model_name, "version": version}

In [ ]:
client = TestClient(app)  # rebuild client so it picks up the new route

resp = client.get("/models/fraud-detector")
print(resp.status_code, resp.json())

resp = client.get("/models/fraud-detector?version=3")
print(resp.status_code, resp.json())

## 1.4 Authentication

Now the part that trips up most people the first time: **you almost never serve a model publicly without some form of auth.** Here are the three schemes you'll encounter constantly.

| Scheme | How it works | Typical use case |
|---|---|---|
| **API Key** | Client sends a static secret in a header | Internal services, simple B2B integrations |
| **HTTP Basic** | Client sends `username:password` base64-encoded in a header | Quick internal tools, rarely production-grade |
| **OAuth2 + JWT** | Client exchanges credentials for a short-lived signed token, sends token on each request | Production APIs, especially multi-user or multi-tenant systems |

We'll implement all three against the same `/predict`-style endpoint so you can compare them directly.

### 1.4.1 API Key auth (header-based)

In [ ]:
from fastapi import FastAPI, Security, HTTPException, status
from fastapi.security import APIKeyHeader

app_key = FastAPI(title="API-Key-protected service")

API_KEY = "supersecret-demo-key"   # in real life: read from env var / secret manager
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verify_api_key(key: str = Security(api_key_header)):
    if key != API_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing API key",
        )
    return key

@app_key.get("/secure-ping")
def secure_ping(key: str = Security(verify_api_key)):
    return {"message": "authenticated with API key"}

In [ ]:
client_key = TestClient(app_key)

# No key -> 401
print(client_key.get("/secure-ping").status_code)

# Wrong key -> 401
print(client_key.get("/secure-ping", headers={"X-API-Key": "wrong"}).status_code)

# Correct key -> 200
resp = client_key.get("/secure-ping", headers={"X-API-Key": "supersecret-demo-key"})
print(resp.status_code, resp.json())

### 1.4.2 HTTP Basic auth

In [ ]:
import secrets
from fastapi import Depends
from fastapi.security import HTTPBasic, HTTPBasicCredentials

app_basic = FastAPI(title="Basic-auth-protected service")
security = HTTPBasic()

def verify_basic(credentials: HTTPBasicCredentials = Depends(security)):
    correct_username = secrets.compare_digest(credentials.username, "ml-engineer")
    correct_password = secrets.compare_digest(credentials.password, "hunter2")
    if not (correct_username and correct_password):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid credentials",
            headers={"WWW-Authenticate": "Basic"},
        )
    return credentials.username

@app_basic.get("/secure-ping")
def secure_ping_basic(user: str = Depends(verify_basic)):
    return {"message": f"hello {user}, you're authenticated"}

In [ ]:
import base64

client_basic = TestClient(app_basic)

# No credentials -> 401
print(client_basic.get("/secure-ping").status_code)

# Correct credentials via TestClient's auth param
resp = client_basic.get("/secure-ping", auth=("ml-engineer", "hunter2"))
print(resp.status_code, resp.json())

### 1.4.3 OAuth2 + JWT (the production pattern)

This is the scheme you'll see wrapping most real model-serving APIs. The flow:

1. Client `POST`s credentials to `/token`
2. Server verifies them and returns a **signed JWT** with an expiry
3. Client sends that JWT as `Authorization: Bearer <token>` on every subsequent request
4. Server verifies the signature + expiry on each request — no database lookup needed per request

This is why it scales well: the token itself carries the proof of identity.

In [ ]:
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm

SECRET_KEY = "demo-signing-secret-do-not-use-in-prod"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Toy user "database"
FAKE_USERS = {"ml-engineer": "hunter2"}

app_jwt = FastAPI(title="JWT-protected service")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

def create_access_token(subject: str, expires_delta: timedelta):
    expire = datetime.now(timezone.utc) + expires_delta
    payload = {"sub": subject, "exp": expire}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

@app_jwt.post("/token")
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    valid_password = FAKE_USERS.get(form_data.username)
    if valid_password is None or valid_password != form_data.password:
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    token = create_access_token(
        subject=form_data.username,
        expires_delta=timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES),
    )
    return {"access_token": token, "token_type": "bearer"}

def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        if username is None:
            raise HTTPException(status_code=401, detail="Invalid token")
        return username
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")

@app_jwt.get("/secure-ping")
def secure_ping_jwt(user: str = Depends(get_current_user)):
    return {"message": f"hello {user}, your JWT is valid"}

In [ ]:
client_jwt = TestClient(app_jwt)

# Step 1: log in to get a token
login_resp = client_jwt.post("/token", data={"username": "ml-engineer", "password": "hunter2"})
print("login:", login_resp.status_code, login_resp.json())

token = login_resp.json()["access_token"]

# Step 2: call the protected endpoint with the bearer token
resp = client_jwt.get("/secure-ping", headers={"Authorization": f"Bearer {token}"})
print("protected call:", resp.status_code, resp.json())

# Step 3: what happens with no token / a garbage token
print("no token:", client_jwt.get("/secure-ping").status_code)
print("bad token:", client_jwt.get("/secure-ping", headers={"Authorization": "Bearer garbage"}).status_code)

### Which one should you actually use?

- **API Key** → good enough for internal service-to-service calls behind a private network (e.g., a feature-store service calling your model service). Simple, but the key is long-lived — rotate it and store it in a secret manager, never in code.
- **HTTP Basic** → fine for quick internal tools / demos; avoid for anything public since credentials go on every request (always pair with HTTPS).
- **OAuth2 + JWT** → the default for anything customer-facing, multi-tenant, or where you need per-user scopes/expiry. This is also the pattern you'll layer under a managed platform like Vertex AI if you're not just using its built-in IAM-based auth (more on that in Module 4).

In managed environments (Vertex AI Endpoints, for example), a lot of this is handled for you via **Google Cloud IAM** — but you still need to know this if you're wrapping the model in your own FastAPI service, e.g., because you need custom pre/post-processing, multiple models behind one gateway, or you're deploying outside GCP.

## Exercise 1 (5–10 min)

Modify the API-key example so that:
1. There are **two** valid API keys, mapped to two different "clients" (`"client_a"`, `"client_b"`), each with a different rate-limit tier (just store it in a dict, no need to enforce rate limiting yet).
2. `/secure-ping` returns which client is calling in the response body.
3. Bonus: raise a `403` (not `401`) if the key is valid but the client is disabled (add a `"disabled": True` flag for one client).

Use the scaffold below.

In [ ]:
# --- Exercise scaffold ---
CLIENTS = {
    "key-for-client-a": {"name": "client_a", "tier": "gold", "disabled": False},
    "key-for-client-b": {"name": "client_b", "tier": "silver", "disabled": True},
}

app_ex = FastAPI()
api_key_header_ex = APIKeyHeader(name="X-API-Key", auto_error=False)

def verify_client(key: str = Security(api_key_header_ex)):
    # TODO: look up `key` in CLIENTS
    # TODO: raise 401 if not found
    # TODO: raise 403 if disabled
    # TODO: return the client dict
    raise HTTPException(status_code=501, detail="not implemented yet")

@app_ex.get("/secure-ping")
def secure_ping_ex(client: dict = Security(verify_client)):
    return {"message": f"hello {client['name']}", "tier": client["tier"]}

# Uncomment once implemented:
# client_ex = TestClient(app_ex)
# print(client_ex.get("/secure-ping", headers={"X-API-Key": "key-for-client-a"}).json())
# print(client_ex.get("/secure-ping", headers={"X-API-Key": "key-for-client-b"}).status_code)  # expect 403
# print(client_ex.get("/secure-ping", headers={"X-API-Key": "nope"}).status_code)               # expect 401

## Recap

- An API is the contract between your model and everything that needs to call it.
- Every HTTP call has a method, path, headers, body, and status code — know the status code table cold.
- FastAPI gives you routing, validation, and docs almost for free; `TestClient` lets you test without spinning up a server.
- Three auth patterns: **API Key** (simple, internal), **HTTP Basic** (quick/internal only), **OAuth2 + JWT** (production default).

**Next up — Module 2:** GET vs POST in depth, request/response schemas with Pydantic, and wrapping an actual ML model behind `/predict`.